# Práctica M35: NumPy y análisis de ventas de supermercado

#### **Objetivo:** aplicar estadística descriptiva, promedios de ticket, rankings y correlaciones utilizando un archivo de ventas de supermercado.

#### **Archivo:** `supermarket_sales.csv` (1,000 transacciones). Se emplea **NumPy** para las operaciones estadísticas y correlaciones, con apoyo de **Pandas** para agrupar los datos.

#### **Nota:** el CSV original utiliza encabezados como `Unit price` y `Customer type`; en la práctica esos campos se mencionan como `unit_price` y `member` respectivamente.

## 1. Importar librerías y cargar los datos

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Colocamos el CSV en la misma carpeta del notebook.
archivo = Path('supermarket_sales.csv')
df = pd.read_csv(archivo)

print('Número de transacciones:', len(df))
print('Columnas:', list(df.columns))
display(df.head())

Número de transacciones: 1000
Columnas: ['Invoice ID', 'Branch', 'City', 'Customer type', 'Gender', 'Product line', 'Unit price', 'Quantity', 'Tax 5%', 'Total', 'Date', 'Time', 'Payment', 'cogs', 'gross margin percentage', 'gross income', 'Rating']


,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3


## 2. Estadística descriptiva: precio unitario y cantidad
Calculamos **media**, **mediana** con NumPy y **moda** mediante conteos con Pandas. Si existen varias modas se muestran todas.

In [2]:
for columna in ['Unit price', 'Quantity']:
    valores = df[columna].dropna().to_numpy()
    media = np.mean(valores)
    mediana = np.median(valores)
    moda = df[columna].mode().tolist()
    print(f'\n{columna}')
    print(f'Media: {media:.2f}')
    print(f'Mediana: {mediana:.2f}')
    print('Moda(s):', moda)


Unit price
Media: 55.67
Mediana: 55.23
Moda(s): [83.77]

Quantity
Media: 5.51
Mediana: 5.00
Moda(s): [10]


## 3. Ticket promedio
Cada fila representa una transacción o *ticket*. Para cada grupo, calculamos el promedio de `Total`, **incluyendo el impuesto del 5%**. Se muestra además la cantidad de tickets del grupo.

In [3]:
def ticket_promedio(columnas):
    return (df.groupby(columnas)['Total']
              .agg(['mean', 'count'])
              .rename(columns={'mean': 'Ticket promedio', 'count': 'Tickets'})
              .round(2))

print('1) Por ciudad')
display(ticket_promedio('City'))

1) Por ciudad


,Ticket promedio,Tickets
City,,
Mandalay,319.87,332
Naypyitaw,337.10,328
Yangon,312.35,340


In [4]:
print('2) Por línea de producto y ciudad')
display(ticket_promedio(['Product line', 'City']))

2) Por línea de producto y ciudad


Ticket promedio  Tickets
Product line           City                               
Electronic accessories Mandalay            310.03       55
                       Naypyitaw           344.89       55
                       Yangon              305.29       60
Fashion accessories    Mandalay            264.73       62
                       Naypyitaw           331.69       65
                       Yangon              320.25       51
Food and beverages     Mandalay            304.30       50
                       Naypyitaw           360.10       66
                       Yangon              295.92       58
Health and beauty      Mandalay            376.99       53
                       Naypyitaw           319.53       52
                       Yangon              268.04       47
Home and lifestyle     Mandalay            350.98       50
                       Naypyitaw           308.79       45
                       Yangon              344.88       65
Sports and travel      Mandalay            322.39       62
                       Naypyitaw           350.27       45
                       Yangon              328.35       59

In [5]:
print('3) Por género')
display(ticket_promedio('Gender'))

3) Por género


,Ticket promedio,Tickets
Gender,,
Female,335.10,501
Male,310.79,499


In [6]:
print('4) Por género y línea de producto')
display(ticket_promedio(['Gender', 'Product line']))

4) Por género y línea de producto


Ticket promedio  Tickets
Gender Product line                                    
Female Electronic accessories           322.64       84
       Fashion accessories              317.06       96
       Food and beverages               368.57       90
       Health and beauty                290.02       64
       Home and lifestyle               380.21       79
       Sports and travel                324.71       88
Male   Electronic accessories           316.69       86
       Fashion accessories              291.08       82
       Food and beverages               273.50       84
       Health and beauty                348.10       88
       Home and lifestyle               294.14       81
       Sports and travel                340.36       78

## 4. Rankings: top 5 en ventas y porcentaje de aporte
Primero sumamos `Total` por categoría, luego aplicamos `rank(method="dense", ascending=False)` y calculamos el porcentaje que representa cada categoría sobre el total de ventas. Si un grupo tiene menos de cinco categorías, se muestran las disponibles.

In [7]:
def top_ventas(columna):
    ventas = df.groupby(columna, as_index=False)['Total'].sum()
    ventas['Rank'] = ventas['Total'].rank(method='dense', ascending=False).astype(int)
    ventas['Aporte (%)'] = 100 * ventas['Total'] / df['Total'].sum()
    return (ventas.sort_values(['Rank', columna])
                  .head(5)
                  .round({'Total': 2, 'Aporte (%)': 2}))

print('1) Top 5: ventas por ciudad')
display(top_ventas('City'))

1) Top 5: ventas por ciudad


,City,Total,Rank,Aporte (%)
1,Naypyitaw,110568.71,1,34.24
2,Yangon,106200.37,2,32.88
0,Mandalay,106197.67,3,32.88


In [8]:
print('2) Top 5: ventas por tipo de cliente (Member/Normal)')
display(top_ventas('Customer type'))

2) Top 5: ventas por tipo de cliente (Member/Normal)


,Customer type,Total,Rank,Aporte (%)
0,Member,164223.44,1,50.85
1,Normal,158743.30,2,49.15


In [9]:
print('3) Top 5: ventas por método de pago')
display(top_ventas('Payment'))

3) Top 5: ventas por método de pago


,Payment,Total,Rank,Aporte (%)
0,Cash,112206.57,1,34.74
2,Ewallet,109993.11,2,34.06
1,Credit card,100767.07,3,31.20


**Interpretación:** `Member` y `Normal` son los dos valores existentes en `Customer type`; no representan clientes individuales. La cantidad de categorías posibles es tres ciudades, dos tipos de cliente y tres medios de pago, así que no pueden existir cinco posiciones distintas en esos rankings.

## 5. Correlaciones con NumPy
Extraemos únicamente la **hora** del campo `Time` (sin minutos) y calculamos Pearson con `np.corrcoef`. También comparamos precio unitario y calificación de la transacción.

In [10]:
df['Hora'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour

matriz_hora_total = np.corrcoef(df['Hora'], df['Total'])
print('Matriz de correlación: hora y total')
display(pd.DataFrame(matriz_hora_total, index=['Hora', 'Total'], columns=['Hora', 'Total']).round(4))
print('Correlación hora-total:', round(matriz_hora_total[0, 1], 4))

Matriz de correlación: hora y total


,Hora,Total
Hora,1.0000,-0.0028
Total,-0.0028,1.0000


Correlación hora-total: -0.0028


In [11]:
matriz_precio_rating = np.corrcoef(df['Unit price'], df['Rating'])
print('Matriz de correlación: precio unitario y rating')
display(pd.DataFrame(matriz_precio_rating, index=['Unit price', 'Rating'], columns=['Unit price', 'Rating']).round(4))
print('Correlación precio-rating:', round(matriz_precio_rating[0, 1], 4))

Matriz de correlación: precio unitario y rating


,Unit price,Rating
Unit price,1.0000,-0.0088
Rating,-0.0088,1.0000


Correlación precio-rating: -0.0088


### Importante: correlación y margen no significan lo mismo

La calificación (`Rating`) refleja satisfacción o valoración de la transacción, **no margen de ganancia**. Por lo tanto, la correlación `Unit price`–`Rating` **no permite determinar** si los productos caros generan mayor margen. El archivo también contiene `gross income`, `cogs` y `gross margin percentage`, que son más apropiados para estudiar resultados económicos. No debemos confundir una relación estadística con causalidad.

## 6. Conclusión breve

In [12]:
print('Resumen del análisis:')
print('Total de tickets:', len(df))
print('Ventas totales:', round(np.sum(df['Total'].to_numpy()), 2))
print('Promedio general por ticket:', round(np.mean(df['Total'].to_numpy()), 2))
print('Correlación hora-total:', round(matriz_hora_total[0,1], 4))
print('Correlación precio-rating:', round(matriz_precio_rating[0,1], 4))

Resumen del análisis:
Total de tickets: 1000
Ventas totales: 322966.75
Promedio general por ticket: 322.97
Correlación hora-total: -0.0028
Correlación precio-rating: -0.0088
